In [ ]:
import pandas as pd

# load corpus

df = pd.read_csv(
    "rp_all_articles.csv",
    low_memory=False
)

print("=" * 80)

print("rows")
print(len(df))

print("\ncolumns")

for c in df.columns:
    print(c)

print("\n" + "=" * 80)

# missing values

summary = []

for col in df.columns:

    missing = df[col].isna().sum()

    empty = 0

    if df[col].dtype == object:

        empty = (

            df[col]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()

        )

    summary.append({

        "column": col,
        "missing": missing,
        "empty_strings": empty,
        "filled": len(df) - max(missing, empty)

    })

summary = pd.DataFrame(summary)

print(summary.sort_values("missing", ascending=False))


In [ ]:
# filter articles about Russia

russia_keywords = [

    # Russia

    r"\brosj\w*\b",

    # Kremlin

    r"\bkreml\w*\b",

    # Moscow

    r"\bmoskw\w*\b",

    # Putin

    r"\bputin\b",

    # Russian officials

    r"\bławrow\w*\b",
    r"\blawrow\w*\b",

    r"\bpieskow\w*\b",

    r"\bszojgu\w*\b",

    r"\bgierasimow\w*\b",
    r"\bgerasimow\w*\b",

    r"\bmiedwiediew\w*\b",

    r"\bzacharow\w*\b",

    r"\bmiszustin\w*\b",

    # Companies

    r"\bgazprom\b",

    r"\brosnieft\b",

    r"\brosatom\b",

    r"\btransnieft\b",

    r"\błukoil\b",
    r"\blukoil\b",

    # Institutions

    r"\bduma\b",

    # Crimea

    r"\bkrym\b",

    # War names

    r"\bwojna (na|w) ukrainie\b",

    r"\bwojna rosji z ukrainą\b",
    r"\bwojna rosji z ukraina\b",

    r"\brosyjska agresja\b",
    r"\bagresja rosji\b"

]
pattern = "|".join(russia_keywords)

russia_df = df[

    (

        df["title"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )

    )

    |

    (

        df["text"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )

    )

].copy()

print()

print("matched articles:")

print(len(russia_df))
# count articles matched by each keyword

keyword_counts = {}

text = (

    df["title"]
    .fillna("")
    + " "
    + df["text"]
    .fillna("")

)

for keyword in russia_keywords:

    keyword_counts[keyword] = (

        text
        .str.contains(
            keyword,
            case=False,
            na=False,
            regex=True
        )
        .sum()

    )

keyword_counts = (

    pd.Series(keyword_counts)
    .sort_values(ascending=False)
    .reset_index()

)

keyword_counts.columns = [

    "keyword",
    "articles"

]

display(keyword_counts)


In [ ]:
# compare with old high-recall corpus

old_df = pd.read_csv(
    "rp_russia_ukraine_corpus.csv",
    low_memory=False
)

old_urls = set(old_df["url"])

new_urls = set(russia_df["url"])

print(f"old corpus: {len(old_urls):,}")

print(f"new corpus: {len(new_urls):,}")
# compare overlap

print(f"overlap: {len(old_urls & new_urls):,}")

print(f"only old: {len(old_urls - new_urls):,}")

print(f"only new: {len(new_urls - old_urls):,}")


In [ ]:
# save final corpus

russia_df.to_csv(

    "rp_SECONDROUND_final_articles.csv",

    index=False,

    encoding="utf-8-sig"

)

print()

print("final corpus saved")

print(len(russia_df))

In [ ]:
# keep only articles that are new compared to the old corpus

only_new_urls = new_urls - old_urls

only_new_df = russia_df[
    russia_df["url"].isin(only_new_urls)
].copy()

print(len(only_new_df))

In [ ]:
from tqdm import tqdm
from bs4 import BeautifulSoup
import requests
import pandas as pd

HEADERS = {
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
}

rows = []

session = requests.Session()

for _, row in tqdm(
    only_new_df.iterrows(),
    total=len(missing_df)
):

    try:

        r = session.get(
            row["url"],
            headers=HEADERS,
            timeout=60
        )

        if r.status_code != 200:
            continue

        soup = BeautifulSoup(
            r.text,
            "lxml"
        )

        title = None
        publish_date = None
        og_image = None

        m = soup.find("meta", property="og:title")
        if m:
            title = m.get("content")

        m = soup.find("meta", property="article:published_time")
        if m:
            publish_date = m.get("content")

        m = soup.find("meta", property="og:image")
        if m:
            og_image = m.get("content")

        text = soup.get_text(" ", strip=True)

        image_urls = []

        for img in soup.find_all("img"):

            url = (
                img.get("src")
                or img.get("data-src")
                or img.get("data-lazy-src")
                or img.get("data-original")
            )

            if url and not url.startswith("data:image"):
                image_urls.append(url)

        rows.append({

            "url": row["url"],
            "category": row["category"],
            "title": title,
            "publish_date": publish_date,
            "text": text,
            "text_length": len(text),
            "og_image": og_image,
            "image_count": len(set(image_urls)),
            "image_urls": "|".join(sorted(set(image_urls)))

        })

    except Exception:

        continue

new_images_df = pd.DataFrame(rows)

print(new_images_df.shape)

new_images_df.to_csv(
    "rp_SECONDROUND_missing_with_images.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
print(len(new_images_df))

In [ ]:
# URLs that were processed successfully

done_urls = set(new_images_df["url"])

# Articles that are still missing

missing_df = only_new_df[
    ~only_new_df["url"].isin(done_urls)
].copy()

print(f"Missing articles: {len(missing_df)}")

In [ ]:
print((new_images_df["image_count"] > 0).sum())

In [ ]:
import os

SOURCE_FOLDER = "."

files = os.listdir(SOURCE_FOLDER)

print(len(files))

print(files[:20])

In [ ]:
import os
import ast
import shutil
import requests
import pandas as pd
from tqdm import tqdm

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

OLD_WITH_IMAGES = (
    "rp_russia_ukraine_with_images.csv"
)

OLD_UNIQUE_IMAGES = (
    "rp_unique_images.csv"
)

OLD_IMAGE_FOLDER = (
    "."
)

NEW_ARTICLES = (
    "rp_SECONDROUND_missing_with_images.csv"
)

FINAL_ARTICLES = (
    "rp_SECONDROUND_final_articles.csv"
)

OUTPUT_FOLDER = (
    "rp_SECONDROUND_images"
)

FAILED_FILE = (
    "rp_SECONDROUND_failed_images.csv"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

old_articles = pd.read_csv(
    OLD_WITH_IMAGES,
    low_memory=False
)

new_articles = pd.read_csv(
    NEW_ARTICLES,
    low_memory=False
)

final_articles = pd.read_csv(
    FINAL_ARTICLES,
    low_memory=False
)

# ------------------------------------------------------------------
# KEEP ONLY OVERLAP FROM OLD CORPUS
# ------------------------------------------------------------------

overlap_urls = set(final_articles["url"]) & set(old_articles["url"])

old_overlap = old_articles[
    old_articles["url"].isin(overlap_urls)
].copy()

print("old overlap:", len(old_overlap))

print("new articles:", len(new_articles))

# ------------------------------------------------------------------
# MERGE
# ------------------------------------------------------------------

merged = pd.concat(
    [
        old_overlap,
        new_articles
    ],
    ignore_index=True
)

merged = merged.drop_duplicates(
    subset="url"
)

print("merged corpus:", len(merged))

merged.to_csv(
    "rp_SECONDROUND_with_images.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------------
# COLLECT UNIQUE IMAGE URLS
# ------------------------------------------------------------------

all_images = set()

for urls in merged["image_urls"].fillna(""):

    text = str(urls).strip()

    if text == "":
        continue

    if text.startswith("["):

        try:

            urls_list = ast.literal_eval(text)

        except Exception:

            continue

    else:

        urls_list = text.split("|")

    for url in urls_list:

        url = str(url).strip()

        if not url.startswith("http"):
            continue

        if "logo_pap" in url:
            continue

        all_images.add(url)

unique_images = pd.DataFrame({

    "image_url": sorted(all_images)

})

unique_images.to_csv(

    "rp_SECONDROUND_unique_images.csv",

    index=False,

    encoding="utf-8-sig"

)

print()

print("unique images:", len(unique_images))

# ------------------------------------------------------------------
# COMPARE WITH OLD UNIQUE IMAGES
# ------------------------------------------------------------------

old_unique = pd.read_csv(
    OLD_UNIQUE_IMAGES
)

old_set = set(
    old_unique["image_url"]
)

new_set = set(
    unique_images["image_url"]
)

already_have = new_set & old_set

need_download = new_set - old_set

print()

print("already downloaded:", len(already_have))

print("need download:", len(need_download))

# ------------------------------------------------------------------
# MAP URL -> FILENAME
# ------------------------------------------------------------------

url_to_filename = {}

for i, url in enumerate(
    old_unique["image_url"]
):

    base = f"{i:07d}"

    for ext in [

        ".jpg",
        ".png",
        ".webp"

    ]:

        filename = base + ext

        path = os.path.join(
            OLD_IMAGE_FOLDER,
            filename
        )

        if os.path.exists(path):

            url_to_filename[url] = filename

            break


In [ ]:
# ------------------------------------------------------------------
# COPY EXISTING IMAGES
# ------------------------------------------------------------------

copied = 0

for url in tqdm(
    already_have,
    desc="Copying"
):

    if url not in url_to_filename:
        continue

    filename = url_to_filename[url]

    shutil.copy2(

        os.path.join(
            OLD_IMAGE_FOLDER,
            filename
        ),

        os.path.join(
            OUTPUT_FOLDER,
            filename
        )

    )

    copied += 1

print()

print("copied:", copied)

# ------------------------------------------------------------------
# DOWNLOAD NEW IMAGES
# ------------------------------------------------------------------

session = requests.Session()

failed = []

downloaded = 0

start_index = len(old_unique)

for i, url in tqdm(

    enumerate(sorted(need_download)),

    total=len(need_download),

    desc="Downloading"

):

    try:

        r = session.get(
            url,
            timeout=60,
            stream=True
        )

        if r.status_code != 200:

            failed.append(url)

            continue

        ext = ".jpg"

        ctype = r.headers.get(
            "Content-Type",
            ""
        ).lower()

        if "png" in ctype:
            ext = ".png"

        elif "webp" in ctype:
            ext = ".webp"

        filename = f"{start_index+i:07d}{ext}"

        with open(

            os.path.join(
                OUTPUT_FOLDER,
                filename
            ),

            "wb"

        ) as f:

            for chunk in r.iter_content(8192):

                f.write(chunk)

        downloaded += 1

    except Exception:

        failed.append(url)

pd.DataFrame({

    "image_url": failed

}).to_csv(

    FAILED_FILE,

    index=False,

    encoding="utf-8-sig"

)

print()

print("downloaded:", downloaded)

print("failed:", len(failed))

In [ ]:
import pandas as pd

INPUT_FILE = "rp_SECONDROUND_final_articles.csv"

df = pd.read_csv(INPUT_FILE, low_memory=False)

unique_urls = df["url"].dropna().unique()

print("total rows:", len(df))
print("unique article urls:", len(unique_urls))

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

INPUT_FILE = "rp_SECONDROUND_final_articles.csv"

df = pd.read_csv(INPUT_FILE, low_memory=False)

sample_urls = df["url"].dropna().sample(50, random_state=42)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
}

results = []

for url in sample_urls:
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            results.append({"url": url, "status": r.status_code, "date": None})
            continue
        soup = BeautifulSoup(r.text, "lxml")
        m = soup.find("meta", attrs={"name": "content:publishDate"})
        results.append({"url": url, "status": r.status_code, "date": m.get("content") if m else None})
    except Exception as e:
        results.append({"url": url, "status": "error", "date": None})

result_df = pd.DataFrame(results)
print(result_df)
print("missing:", result_df["date"].isna().sum())

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

INPUT_FILE = "rp_SECONDROUND_final_articles.csv"
OUTPUT_FILE = "rp_SECONDROUND_final_articles.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
}

df = pd.read_csv(INPUT_FILE, low_memory=False)

urls = df["url"].dropna().unique()

def fetch_date(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            return url, None
        soup = BeautifulSoup(r.text, "lxml")
        m = soup.find("meta", attrs={"name": "content:publishDate"})
        return url, m.get("content") if m else None
    except Exception:
        return url, None

results = {}

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(fetch_date, url) for url in urls]
    for future in tqdm(as_completed(futures), total=len(futures)):
        url, date = future.result()
        results[url] = date

df["publish_date"] = df["url"].map(results)

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("missing dates:", df["publish_date"].isna().sum())


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

INPUT_FILE = "rp_SECONDROUND_final_articles.csv"
OUTPUT_FILE = "rp_SECONDROUND_final_articles.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
}

MAX_RETRIES = 3

df = pd.read_csv(INPUT_FILE, low_memory=False)

missing_urls = df.loc[df["publish_date"].isna(), "url"].dropna().unique()
print("urls to retry:", len(missing_urls))

def fetch_date(url):
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code != 200:
                time.sleep(2)
                continue
            soup = BeautifulSoup(r.text, "lxml")
            m = soup.find("meta", attrs={"name": "content:publishDate"})
            if m:
                return url, m.get("content")
            return url, None
        except Exception:
            time.sleep(2)
    return url, None

results = {}

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(fetch_date, url) for url in missing_urls]
    for future in tqdm(as_completed(futures), total=len(futures)):
        url, date = future.result()
        results[url] = date

df["publish_date"] = df["publish_date"].fillna(df["url"].map(results))

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("still missing dates:", df["publish_date"].isna().sum())